In [ ]:
import squidpy as sq
import scanpy as sc
import scipy as scp
import numpy as np 
import pandas as pd
import scvi
import seaborn as sns
import pyensembl as ensb


In [ ]:
normalCells = sc.read_h5ad('hlca/nor.h5ad')
sclcCells = sc.read_h5ad('hlca/sclcCells.h5ad')

In [ ]:
sc.pp.filter_genes(sclcCells, min_cells=10)
sc.pp.highly_variable_genes(sclcCells, n_top_genes=2000, subset=True, flavor='seurat_v3') # using seurat_v3 because of raw counts

In [ ]:
sclcCells

In [ ]:
scvi.model.SCVI.setup_anndata(sclcCells)
vae = scvi.model.SCVI(sclcCells)
vae.train()

In [ ]:
solo = scvi.external.SOLO.from_scvi_model(vae) # get the SOLO model from SCVI
solo.train() # train the model over the standard number of epochs, set to auto-terminate

In [ ]:
doubletPredict = solo.predict() # use the SOLO model to predict if a cell is a singlet or doublet
doubletPredict['prediction'] = solo.predict(soft=False) # predict in a new column
doubletPredict.head(5) # visualise counts

In [ ]:
doubletPredict.groupby('prediction').count() # doublets - 126, singlets - 20505


In [ ]:
doubletPredict['predDif'] = abs(doubletPredict['singlet'] - doubletPredict['doublet']) # visualise certainty with which singlets are predicted
sns.displot(doubletPredict[doubletPredict.prediction == 'doublet'], x = 'predDif')
doubletPredict.shape 

In [ ]:
doublets = doubletPredict[(doubletPredict.prediction == 'doublet') & (doubletPredict.predDif > 1)]

In [ ]:
sclcCells_Full = sc.read_h5ad('hlca/sclcCells.h5ad')
sclcCells_Full.obs['doublet'] = sclcCells_Full.obs.index.isin(doublets.index)

In [ ]:
sclcCells_Full.obs

In [ ]:
sclcCells_Full = sclcCells_Full[~sclcCells_Full.obs.doublet]

In [ ]:
sclcCells_Full.write_h5ad('sclcSinglets.h5ad')